# 03. Export features and report usable samples

Compute the existing feature recipes, retain the per-parcel cap and locked partitions, and save separate development and CropHarvest external-test packages. The resulting tables show how many fields and pixels passed, separately by crop and partition. No model fitting or test scoring occurs.


In [ ]:
from pathlib import Path
import sys
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import preprocessing_workflow as workflow
from crop_pipeline.notebook_display import show_table
from crop_pipeline.windows import seasons

window_input = ""  # Optional exact windows.json path from new notebook 02; blank uses its verified pointer.


## Export model inputs

Matched fixed/aligned cohorts retain identical locations. Full cohorts report their own coverage. Budgets are collection targets, not accuracy criteria; unmet budgets and absent crops remain visible. Do not add counts across cohorts.


In [ ]:
feature_handoff = workflow.export_features(root, window_input or None)
usable_samples = seasons.read_table(feature_handoff.parent / "usable_samples.csv")
for cohort in ("fixed_full", "aligned_full", "fixed_matched", "aligned_matched"):
    show_table([r for r in usable_samples if r["cohort"] == cohort], title=f"Usable samples: {cohort}")


## Usable samples against targets

Dashed lines are the per-crop targets: 60 parcels / 1,200 pixels for training and 20 parcels / 400 pixels each for validation and testing. Bars below a line are shortfalls. Reaching a target does not establish accuracy.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from crop_pipeline.visualization import workflow_figures as figures


def show_and_save(figure, folder, name):
    """Display inline and keep a PNG beside the run outputs for reports and slides."""
    print("Saved figure:", figures.save_figure(figure, folder, name))
    display(figure)
    plt.close(figure)

figure_folder = feature_handoff.parent / "figures"
for cohort in ("fixed_full", "aligned_full", "fixed_matched", "aligned_matched"):
    show_and_save(figures.draw_usable_vs_targets(usable_samples, cohort), figure_folder, f"usable_vs_targets_{cohort}")

## Training and external-test handoffs

Continue with Training 01–02, then Testing 01–02. The notebooks use the current feature export by default. Select models using validation results only.


In [ ]:
print("Development:", feature_handoff.parent / "features_development.json")
print("External test:", feature_handoff.parent / "features_test.json")
print("Readable counts report:", feature_handoff.parent / "preprocessing_report.md")
print("Counts CSV:", feature_handoff.parent / "usable_samples.csv")
print("Organized outputs:", root / "pipeline_outputs/preprocessing" / feature_handoff.parent.name / "features")
